In [12]:
dir_path = "/home/chanhui-lee/text-mol/MolCA/data/multi_task_0927/raw"
test_files = [
    "smol-forward_synthesis_subtask-0_test.pth",
    "smol-molecule_generation_subtask-0_test.pth",
]

# load the testsets
import torch
import os
import numpy as np
import pandas as pd
import random
import selfies as sf

def load_testset(dir_path, test_files):

    testset = []
    for f in test_files:
        list_of_instances = torch.load(os.path.join(dir_path, f))
        list_of_dicts = []
        for instance in list_of_instances:
            dict_instance = {
                "graph": instance[0],
                "label_selfies": instance[1],
                "input_mol_selfies": instance[2],
                "task_subtask_pair": instance[3],
                "instruction": instance[4],
                "conditions": None,
            }
            label_selfies = instance[1].replace("<SELFIES>", "").replace("</SELFIES>", "")
            label_smiles = sf.decoder(label_selfies)
            input_mol_selfies = instance[2].replace("<SELFIES>", "").replace("</SELFIES>", "")
            if "None" in input_mol_selfies:
                input_mol_smiles = input_mol_selfies
            else:
                input_mol_smiles = sf.decoder(input_mol_selfies)

            dict_instance.update({
                "label_selfies": label_selfies,
                "label_smiles": label_smiles,
                "input_mol_selfies": input_mol_selfies,
                "input_mol_smiles": input_mol_smiles,
            })
            list_of_dicts.append(dict_instance)
        random.shuffle(list_of_dicts)
        testset.append(list_of_dicts[:1000])

    return testset

test_paths = [os.path.join(dir_path, f) for f in test_files]
testset = load_testset(dir_path, test_files)
len(testset[0]), len(testset[1])

(1000, 1000)

In [13]:
testset[0][0], testset[1][0]

({'graph': Data(x=[67, 9], edge_index=[2, 134], edge_attr=[134, 3]),
  'label_selfies': '[C][C][O][C][=Branch1][C][=O][N][C][C][C@@H1][Branch2][Ring2][Branch2][N][S][=Branch1][C][=O][=Branch1][C][=O][C][=C][C][=C][Branch1][#C][N][C][=Branch1][C][=O][C][=C][C][=C][C][=C][Ring1][=Branch1][C][C][=C][C][=C][C][=C][Ring2][Ring1][Ring2][Ring1][=Branch1][C@H1][Branch1][Ring1][C][O][C][Ring2][Ring1][S]',
  'input_mol_selfies': '[C][C][C][O][C][Ring1][Branch1].[C][C][Branch1][C][C][Branch1][C][C][O][C][=Branch1][C][=O][N][C][C][C][Branch1][C][N][C][C][Ring1][#Branch1].[C][C][Branch1][C][C][N][=C][=O].[C][C][O][C][=Branch1][C][=O][C@@H1][C][N][Branch1][Branch2][C][=Branch1][C][=O][O][C][C][C][C][C@H1][Ring1][O][N][S][=Branch1][C][=O][=Branch1][C][=O][C][=C][C][=C][Branch1][#C][N][C][=Branch1][C][=O][C][=C][C][=C][C][=C][Ring1][=Branch1][C][C][=C][C][=C][C][=C][Ring2][Ring1][Ring2][Ring1][=Branch1].[BH4-1].[Li+1]',
  'task_subtask_pair': 'smol-forward_synthesis/smol-forward_synthesis',
  'instruc

In [14]:
# save the test_data_1k
test_files_1k = [
    "smol-forward_synthesis_subtask-0_test_1k.pth",
    "smol-molecule_generation_subtask-0_test_1k.pth",
]
os.makedirs(os.path.join(dir_path, "molgen_poc"), exist_ok=True)

for f in test_files_1k:
    for i, test in enumerate(testset):
        torch.save(test, os.path.join(dir_path, "molgen_poc", f))

In [15]:
def get_label_smiles_list(testset):
    total_label_smiles_list = []
    for test in testset:
        label_smiles_list = []
        for instance in test:
            label_smiles_list.append(instance["label_smiles"])
        total_label_smiles_list.append(label_smiles_list)
    return total_label_smiles_list

testset_label_smiles_list = get_label_smiles_list(testset)

In [16]:
len(testset_label_smiles_list), len(testset_label_smiles_list[0]), len(testset_label_smiles_list[1])

(2, 1000, 1000)

In [17]:
# save list as csv
for i, f in enumerate(test_files_1k):
    pd.DataFrame(testset_label_smiles_list[i]).to_csv(os.path.join(dir_path, "molgen_poc", f.replace(".pth", ".csv")), index=False, header=False)